# Train Model with DPO

### imports

In [ ]:
%%capture
%pip install -q datasets pandas matplotlib packaging huggingface_hub


### Load the public preference dataset
No Hugging Face login is required for this public dataset unless you later decide to push checkpoints to the Hub.


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display

os.environ["TOKENIZERS_PARALLELISM"] = "false"

if not torch.cuda.is_available():
    raise RuntimeError(
        "Problem 3 requires a CUDA-enabled Linux GPU environment. "
        "Run this notebook on the HPC GPU node before executing the remaining cells."
    )

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))
print("Hugging Face login is not required for this public dataset.")


In [ ]:
dataset = load_dataset("EliasHossain/youtube-titles-dpo")

summary_df = pd.DataFrame(
    [
        {"split": split_name, "num_rows": len(split_data), "columns": list(split_data.column_names)}
        for split_name, split_data in dataset.items()
    ]
)
display(summary_df)

sample_row = dataset["train"][0]
print("Sample keys:", list(sample_row.keys()))
print("Prompt example:", sample_row["prompt"])
print("Chosen example:", sample_row["chosen"])
print("Rejected example:", sample_row["rejected"])


### load model

We are going to use the Unsloth library for efficient 4-bit loading and LoRA fine-tuning. Run this notebook on a CUDA GPU node before continuing.


In [ ]:
MODEL_NAME = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048
OUTPUT_DIR = "./qwen3_youtube_titles_dpo"
TRAIN_SUBSET_SIZE = 256
EVAL_SUBSET_SIZE = 64
BASELINE_SAMPLE_COUNT = 3
MAX_NEW_TOKENS = 64

print("Model name:", MODEL_NAME)
print("Output directory:", OUTPUT_DIR)
print("Train subset size:", TRAIN_SUBSET_SIZE)
print("Eval subset size:", EVAL_SUBSET_SIZE)


Install the required libraries for Unsloth, TRL, PEFT, and bitsandbytes. This cell now uses the simpler Linux/CUDA install path so newer torch builds on HPC nodes do not get blocked by a stale manual version map. Leave the output visible so dependency failures are easy to debug.


In [ ]:
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required before installing the GPU training stack.")

cuda_version = str(torch.version.cuda)
device_name = torch.cuda.get_device_name(0)

print("GPU:", device_name)
print("Torch version:", torch.__version__)
print("CUDA version:", cuda_version)
print("Installing the latest Unsloth build for the current Linux/CUDA environment.")

commands = [
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "packaging", "setuptools", "wheel"],
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        "--no-deps",
        "git+https://github.com/unslothai/unsloth-zoo.git",
    ],
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        "--no-build-isolation",
        "unsloth",
    ],
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "trl==0.9.3",
        "peft",
        "accelerate",
        "bitsandbytes",
        "triton",
    ],
]

for command in commands:
    print("+", " ".join(command))
    subprocess.run(command, check=True)

print("Install cell finished. If the kernel still cannot import Unsloth, restart the kernel and rerun from the imports cell.")


In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import DPOConfig, DPOTrainer
import bitsandbytes as bnb
import pandas as pd
import torch

print("Unsloth import successful.")
print("bitsandbytes version:", getattr(bnb, "__version__", "unknown"))
print("Torch version:", torch.__version__)


In [ ]:
!pip install -U bitsandbytes


In [ ]:
from unsloth import FastLanguageModel
import torch

model_name = MODEL_NAME

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loaded model:", model_name)
print("Tokenizer pad token:", tokenizer.pad_token)


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
def count_trainable_parameters(model):
    trainable_params = 0
    total_params = 0
    for parameter in model.parameters():
        total_params += parameter.numel()
        if parameter.requires_grad:
            trainable_params += parameter.numel()

    percentage = 100 * trainable_params / total_params
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable percentage: {percentage:.4f}%")


model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

count_trainable_parameters(model)


### Generate titles with the base model
Run a small before-training qualitative check on held-out validation prompts.


In [ ]:
def format_chat_prompt(user_input, system_message=None, add_generation_prompt=True):
    """
    Format user input or message lists with the tokenizer chat template.

    Args:
        user_input: Either a raw string or a list of role/content dictionaries.
        system_message: Optional system message to prepend when user_input is a raw string.
        add_generation_prompt: Whether to start the assistant turn.

    Returns:
        str: Formatted prompt for the model.
    """
    if isinstance(user_input, list):
        messages = list(user_input)
    else:
        messages = []
        if system_message:
            messages.append({"role": "system", "content": system_message})
        messages.append({"role": "user", "content": user_input})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
    )


def prepare_dpo_example(example):
    return {
        "prompt": format_chat_prompt(example["prompt"], add_generation_prompt=True),
        "chosen": example["chosen"][-1]["content"],
        "rejected": example["rejected"][-1]["content"],
    }


def generate_title(model, prompt_text, tokenizer_obj=None, max_new_tokens=MAX_NEW_TOKENS):
    tokenizer_obj = tokenizer if tokenizer_obj is None else tokenizer_obj
    inputs = tokenizer_obj(prompt_text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        use_cache=True,
    )
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer_obj.decode(generated_tokens, skip_special_tokens=True).strip()


processed_dataset = dataset.map(
    prepare_dpo_example,
    remove_columns=dataset["train"].column_names,
)

train_subset = processed_dataset["train"].shuffle(seed=42).select(
    range(min(TRAIN_SUBSET_SIZE, len(processed_dataset["train"])))
)
eval_subset = processed_dataset["valid"].shuffle(seed=42).select(
    range(min(EVAL_SUBSET_SIZE, len(processed_dataset["valid"])))
)

comparison_examples = eval_subset.select(range(min(BASELINE_SAMPLE_COUNT, len(eval_subset))))

display(train_subset.to_pandas().head(3))
print("Train subset size:", len(train_subset))
print("Eval subset size:", len(eval_subset))
print("Comparison sample count:", len(comparison_examples))


In [ ]:
FastLanguageModel.for_inference(model)

base_rows = []
for row in comparison_examples:
    base_rows.append(
        {
            "prompt": row["prompt"],
            "chosen_reference": row["chosen"],
            "rejected_reference": row["rejected"],
            "base_generation": generate_title(model, row["prompt"]),
        }
    )

base_results = pd.DataFrame(base_rows)
display(base_results)


### Train the model with DPO
Use a short demonstrator run that is realistic for homework while still producing observable before/after behavior.


In [ ]:
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    max_steps=60,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
    warmup_ratio=0.1,
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    optim="paged_adamw_8bit",
    max_length=1024,
    max_prompt_length=512,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=eval_subset,
    processing_class=tokenizer,
)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete.")
print("Saved adapter to:", OUTPUT_DIR)


### Review training metrics and compare the fine-tuned model


In [ ]:
metrics_df = pd.DataFrame(trainer.state.log_history)
display(metrics_df.tail(10))

metric_columns = [
    "loss",
    "eval_loss",
    "rewards/chosen",
    "rewards/rejected",
    "rewards/accuracies",
    "rewards/margins",
]
available_metric_columns = [column for column in metric_columns if column in metrics_df.columns]

if available_metric_columns:
    metrics_df[available_metric_columns].plot(subplots=True, figsize=(10, 2.5 * len(available_metric_columns)))
    plt.tight_layout()
    plt.show()

try:
    ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
        model_name=OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        load_in_8bit=False,
        full_finetuning=False,
    )
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
    ft_tokenizer.padding_side = "right"
    print("Reloaded fine-tuned model from disk.")
except Exception as error:
    print("Falling back to the in-memory trained model because reload failed:")
    print(error)
    ft_model = trainer.model
    ft_tokenizer = tokenizer

FastLanguageModel.for_inference(ft_model)


In [ ]:
fine_tuned_generations = [
    generate_title(ft_model, row["prompt"], tokenizer_obj=ft_tokenizer)
    for row in comparison_examples
]

comparison_df = base_results.copy()
comparison_df["dpo_generation"] = fine_tuned_generations

display(comparison_df)

print("Discussion guide:")
print("- Compare the base and DPO generations against the chosen titles.")
print("- Check whether the DPO outputs sound more engaging and more closely aligned with the preferred examples.")
print("- Use the rewards and loss curves above to explain any visible improvement or instability.")
